# Zn universal encoder + ExpertXAS architecture sweep

Exploratory study. Not a paper reproduction.

Plan:
1. Build or verify the shared Zn dataset (same source and rules as `train_zn_expert_and_full.ipynb`).
2. Train the nine-element (Ti..Cu, Zn) universal M3GNetXAS encoder once, from scratch, and export the frozen 64D features.
3. Sweep many ExpertXAS head architectures on the **frozen universal-encoder features**. Every architecture uses the same hyperparameters as the Zn notebook's expert branch; only `hidden_dims` changes.
4. Select the winner by validation median MSE. Touch the test split once, for the winner only.
5. For the winner, plot per-spectrum test MSE (x) against a per-spectrum noise ratio (y). The noise ratio is `std(y - gaussian_smooth(y)) / std(y)` with sigma = 0.5 eV.

Rules and warnings:
- The Zn targets are raw-interpolated and unscaled (scale 1). Exported features keep the pipeline's factor-of-1000 scale. This is not the published Figshare pipeline.
- Splits stay material-level; shared Zn materials keep their eight-element split assignment.
- Existing run or dataset directories are never overwritten. The encoder run fails if its directory already exists; delete it yourself to retrain.
- The sweep writes one subdirectory per architecture under `expert_sweep/`. Reruns skip architectures that already have a `best.pt` and `metrics.json`, unless `FORCE_RESWEEP = True`.
- The test split is used only for the selected winner (the cell "Test evaluation and noise-ratio chart").
- Set `CUDA_VISIBLE_DEVICES` before Jupyter starts. Use Python 3.11 and the pinned project packages. Training defaults to off. Set `RUN_TRAINING = True` when ready to train.
- Data comes from the same place as the other committed notebooks: `$OMNIXAS_DATA_ROOT` if set, otherwise `<repo parent>/OmniXAS_data`. The raw Zn spectra and the eight-element unscaled splits must be present there; this machine may not have them, which is fine, the checks fail clearly.


In [ ]:
from pathlib import Path
import hashlib
import json
import math
import os
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "omnixas").is_dir())
sys.path[:0] = [str(REPO_ROOT), str(REPO_ROOT / "tutorial_omnixas")]
DATA_ROOT = Path(os.environ.get("OMNIXAS_DATA_ROOT", REPO_ROOT.parent / "OmniXAS_data")).expanduser().resolve()
RAW_ROOT = DATA_ROOT / "figshare"
BASE_ROOT = DATA_ROOT / "figshare_141_splits_unscaled"
DATASET_ROOT = DATA_ROOT / "figshare_141_splits_unscaled_zn9"
OUTPUT_ROOT = REPO_ROOT / "output/training/zn_expert_sweep"
ENCODER_RUN_NAME = "zn9_universal_encoder"
ENCODER_RUN = OUTPUT_ROOT / ENCODER_RUN_NAME
SWEEP_DIR = OUTPUT_ROOT / "expert_sweep"

ELEMENTS = ["Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu"]
SPLITS = ("train", "val", "test")
SEED = 42
ZN_START_EV = 9659.0
ENERGY_GRID = np.linspace(ZN_START_EV, ZN_START_EV + 35, 141)
STEP_EV = float(ENERGY_GRID[1] - ENERGY_GRID[0])
SIGMA_EV = 0.5  # Gaussian sigma for the noise-ratio residual
RUN_TRAINING = False  # Set True only when ready for the long GPU run
FORCE_RESWEEP = False

# ExpertXAS sweep architectures. Same hyperparameters for all; only hidden_dims change.
# Includes the Cu FEFF default [600, 600, 400] and the UniversalXAS widths [500, 500, 550] as references.
EXPERT_SWEEP = [
    (512,), (1024,), (2048,), (4096,), (8192,),
    (128, 128), (256, 256), (512, 512), (1024, 1024), (2048, 2048), (1024, 2048), (2048, 1024),
    (128, 128, 128), (256, 256, 256), (384, 384, 384), (512, 512, 512), (768, 768, 768),
    (1024, 1024, 1024), (1500, 1500, 1500), (600, 600, 400), (500, 500, 550), (400, 600, 600),
    (256, 512, 256), (1024, 512, 1024), (128, 512, 128), (3000, 1500, 750), (750, 1500, 3000),
    (256, 512, 1024, 512, 256), (512, 1024, 2048, 1024, 512), (64, 256, 1024, 256, 64),
]
def arch_slug(dims):
    return "x".join(str(d) for d in dims)
assert len({arch_slug(d) for d in EXPERT_SWEEP}) == len(EXPERT_SWEEP)

# Fixed hyperparameters, identical to the Zn notebook's expert branch.
HEAD_LR = 0.002
HEAD_DROPOUT = 0.05
HEAD_EPOCHS = 800
HEAD_PATIENCE = 60
HEAD_BATCH = 4096
HEAD_SEED = 43
HEAD_LOSS_BALANCE_POWER = 1.5  # no-op for a single task; kept for parity

print("repo root:", REPO_ROOT)
print("data root:", DATA_ROOT)
print("dataset root:", DATASET_ROOT)
print("output root:", OUTPUT_ROOT)
print("architectures in sweep:", len(EXPERT_SWEEP))


In [ ]:
def read_split(root, element, split):
    ids = (root / "material_id_and_site" / f"{element}_FEFF_{split}.txt").read_text(encoding="utf-8-sig").splitlines()
    keys = [(mid, int(site)) for mid, site in (value.rsplit("_", 1) for value in ids)]
    y = np.loadtxt(root / "ml_data" / f"{element}_FEFF_{split}_y.txt", dtype=np.float32, ndmin=2)
    if not ids or len(keys) != len(set(keys)) or any(not mid or site < 0 for mid, site in keys):
        raise ValueError(f"Empty, duplicate, or invalid IDs: {element} {split}")
    if y.shape != (len(ids), 141) or not np.isfinite(y).all() or (y < 0).any():
        raise ValueError(f"Invalid targets: {element} {split}, {y.shape}")
    return ids, y


def material_splits(data):
    assignments = {}
    for (element, split), (ids, _) in data.items():
        for value in ids:
            material = value.rsplit("_", 1)[0]
            if assignments.setdefault(material, split) != split:
                raise ValueError(f"Material leakage across elements: {material}")
    return assignments


def assign_zn(materials, fixed, seed=SEED):
    materials = sorted(set(materials))
    if len(materials) < 3:
        raise ValueError("Zn needs at least three materials")
    n = len(materials)
    train = min(n - 2, max(1, round(0.8 * n)))
    val = min(n - train - 1, max(1, round(0.1 * n)))
    targets = np.array([train, val, n - train - val])
    assigned = {m: fixed[m] for m in materials if m in fixed}
    counts = np.array([list(assigned.values()).count(s) for s in SPLITS])
    new = np.array([m for m in materials if m not in assigned], dtype=object)
    np.random.default_rng(seed).shuffle(new)
    for material in new:
        index = int(np.argmax(targets - counts))
        assigned[material] = SPLITS[index]
        counts[index] += 1
    if (counts == 0).any():
        raise ValueError("Fixed memberships leave an empty Zn split")
    return assigned


def sample_zn(path):
    table = np.loadtxt(path, comments="#", ndmin=2)
    if table.shape[1] < 4:
        raise ValueError(f"Expected at least four spectrum columns: {path}")
    energy, mu = table[:, 0], table[:, 3]
    keep = np.isfinite(energy) & np.isfinite(mu) & (mu > 0)
    energy, mu = energy[keep], mu[keep]
    if not len(energy):
        raise ValueError(f"No finite positive spectrum values: {path}")
    order = np.argsort(energy, kind="stable")
    energy, mu = energy[order], mu[order]
    energy, starts, counts = np.unique(energy, return_index=True, return_counts=True)
    mu = np.add.reduceat(mu, starts) / counts
    inside = (energy > ENERGY_GRID[0]) & (energy < ENERGY_GRID[-1])
    if not inside.any():
        return np.full(141, mu[np.argmin(abs(energy - ENERGY_GRID[0]))], dtype=np.float32)
    return np.interp(ENERGY_GRID, energy[inside], mu[inside]).astype(np.float32)


In [ ]:
data = {(e, s): read_split(BASE_ROOT, e, s) for e in ELEMENTS for s in SPLITS}
fixed = material_splits(data)
base_manifest = pd.read_csv(BASE_ROOT / "manifest.csv")
for (element, split), (ids, _) in data.items():
    rows = base_manifest.query("element == @element and split == @split").sort_values("row_index")
    if rows.material_id_and_site.tolist() != ids or rows.row_index.tolist() != list(range(len(ids))):
        raise ValueError(f"Source manifest order differs: {element} {split}")
    if not rows.target_scale_factor.eq(1).all():
        raise ValueError("The eight-element targets must use scale 1")

manifest = pd.read_csv(RAW_ROOT / "manifest.csv").query("element == 'Zn'").copy()
valid = manifest.status.eq("matched") & manifest.raw_candidate_count.eq(1)
omitted = manifest.loc[~valid].copy()
zn = manifest.loc[valid].copy()
if zn.empty or zn.duplicated(["material_id", "site"]).any():
    raise ValueError("Missing or duplicate matched Zn records")
sites = pd.to_numeric(zn.site, errors="raise")
if (sites < 0).any() or (sites != np.floor(sites)).any():
    raise ValueError("Zn site indices must be nonnegative integers")
zn["site"] = sites.astype(int)
zn = zn.sort_values(["material_id", "site"])
assignment = assign_zn(zn.material_id, fixed)
zn["split"] = zn.material_id.map(assignment)
zn["id"] = [f"{r.material_id}_{r.site:03d}" for r in zn.itertuples()]
targets = []
for row in tqdm(zn.itertuples(), total=len(zn), desc="Zn spectra"):
    expected = f"FEFF/Zn/{row.material_id}/FEFF-XANES/{row.site:03d}_Zn"
    if row.site_directory != expected:
        raise ValueError(f"Unexpected Zn source directory: {row.site_directory}")
    targets.append(sample_zn(RAW_ROOT / expected / "spectrum_raw.dat"))
targets = np.stack(targets)
for split in SPLITS:
    mask = zn.split.eq(split).to_numpy()
    data["Zn", split] = (zn.loc[mask, "id"].tolist(), targets[mask])
material_splits(data)  # Global check includes all nine elements.
print(f"Excluded {len(omitted)} unmatched or ambiguous Zn records. Shared materials: {sum(m in fixed for m in assignment)}")
display(pd.DataFrame([
    {"element": e, "split": s, "spectra": len(ids), "materials": len({i.rsplit('_', 1)[0] for i in ids})}
    for (e, s), (ids, _) in data.items()
]).pivot(index="element", columns="split", values=["spectra", "materials"]))


In [ ]:
def sha256(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


source_files = [RAW_ROOT / "manifest.csv", BASE_ROOT / "manifest.csv"]
source_files += [BASE_ROOT / folder / f"{e}_FEFF_{s}{suffix}.txt"
                 for e in ELEMENTS for s in SPLITS
                 for folder, suffix in (("material_id_and_site", ""), ("ml_data", "_y"))]
dataset_info = {
    "source": str(RAW_ROOT), "base": str(BASE_ROOT), "seed": SEED,
    "zn_grid_ev": ENERGY_GRID.tolist(), "target_scale": 1, "feature_scale": 1000,
    "sampling": "positive column 4; average duplicate energies; strict interior window; endpoint padding",
    "split_policy": "preserve shared memberships; fill 80/10/10 Zn material-count deficits",
    "omitted_zn_records": len(omitted), "source_sha256": {str(p): sha256(p) for p in source_files},
    "zn_target_sha256": hashlib.sha256(targets.tobytes()).hexdigest(),
}
marker = DATASET_ROOT / "dataset.json"
if DATASET_ROOT.exists():
    if not marker.is_file() or json.loads(marker.read_text()) != dataset_info:
        raise ValueError(f"Existing dataset is incomplete or differs. Use a new DATASET_ROOT: {DATASET_ROOT}")
else:
    for folder in ("material_id_and_site", "ml_data"):
        (DATASET_ROOT / folder).mkdir(parents=True, exist_ok=True)
    for (element, split), (ids, y) in data.items():
        for folder, suffix in (("material_id_and_site", ""), ("ml_data", "_y")):
            path = DATASET_ROOT / folder / f"{element}_FEFF_{split}{suffix}.txt"
            if element != "Zn":
                source = BASE_ROOT / folder / path.name
                try:
                    path.symlink_to(source)
                except OSError:
                    os.link(source, path)  # Windows fallback. No data copy.
            elif suffix:
                np.savetxt(path, y, fmt="%.9g")
            else:
                path.write_text("".join(i + "\n" for i in ids), encoding="utf-8")
    omitted.to_csv(DATASET_ROOT / "omitted_zn_sites.csv", index=False)
    zn[["material_id", "site", "id", "split", "site_directory"]].to_csv(DATASET_ROOT / "zn_manifest.csv", index=False)
    marker.write_text(json.dumps(dataset_info, indent=2), encoding="utf-8")

for (element, split), (ids, y) in data.items():
    saved_ids, saved_y = read_split(DATASET_ROOT, element, split)
    if saved_ids != ids or not np.array_equal(saved_y, y):
        raise ValueError(f"Stored dataset differs: {element} {split}")
print(f"Dataset checked: {DATASET_ROOT}")


In [ ]:
def gaussian_kernel(sigma_ev, step_ev=STEP_EV, radius=3):
    sigma_pt = sigma_ev / step_ev
    half = max(1, int(radius * sigma_pt))
    x = np.arange(-half, half + 1)
    k = np.exp(-0.5 * (x / sigma_pt) ** 2)
    return k / k.sum()


def smooth_spectrum(y, sigma_ev=SIGMA_EV):
    k = gaussian_kernel(sigma_ev, STEP_EV)
    half = len(k) // 2
    y_pad = np.pad(y, half, mode="edge")
    return np.convolve(y_pad, k, mode="valid")


def noise_ratio(y, sigma_ev=SIGMA_EV):
    """std of the high-frequency residual divided by std of the spectrum.

    residual = y - gaussian_smooth(y); sigma = 0.5 eV by default.
    Returns 0.0 for a constant spectrum.
    """
    y = np.asarray(y, dtype=np.float64)
    scale = np.std(y)
    if scale == 0.0:
        return 0.0
    return float(np.std(y - smooth_spectrum(y, sigma_ev)) / scale)


def spearman(a, b):
    from scipy.stats import spearmanr
    return float(spearmanr(a, b).statistic)


def score_predictions(prediction, y, train_y):
    if prediction.shape != y.shape or not np.isfinite(prediction).all():
        raise ValueError("Invalid predictions")
    errors = ((prediction - y) ** 2).mean(1)
    baseline = float(np.median(((y - train_y.mean(0)) ** 2).mean(1)))
    median = float(np.median(errors))
    eta = baseline / median if median > 0 else (float("inf") if baseline > 0 else float("nan"))
    return {"mse": float(errors.mean()), "median_mse": median, "baseline_median_mse": baseline, "eta": eta}


# Small deterministic checks. No training, no data files touched.
fixed_check = {"shared": "test"}
assigned_check = assign_zn(["shared", *[f"new-{i}" for i in range(99)]], fixed_check)
assert assigned_check["shared"] == "test"
assert [list(assigned_check.values()).count(s) for s in SPLITS] == [80, 10, 10]
assert assigned_check == assign_zn(list(reversed(assigned_check)), fixed_check)
x = np.linspace(0, 4 * np.pi, 141)
smooth_signal = np.sin(x) + 2.0
rng = np.random.default_rng(0)
noisy = smooth_signal + rng.normal(0, 0.5, size=x.shape)
assert noise_ratio(smooth_signal) < 0.05
assert noise_ratio(noisy) > noise_ratio(smooth_signal)
assert noise_ratio(np.full(141, 3.0)) == 0.0
assert np.isclose(gaussian_kernel(1.0).sum(), 1.0)
print("Split, noise-ratio and kernel checks passed.")


## Train the universal encoder

Trains the nine-element M3GNetXAS encoder from scratch (Ti..Cu + Zn) using the same pipeline settings as the committed Zn notebook: 64 encoder rows per element, bf16-mixed, balanced batches that keep every training row, seed 42, macro-relative-median validation early stopping. Only the encoder and its temporary training head are trained in this stage.

The encoder run lands in `output/training/zn_expert_sweep/zn9_universal_encoder/`. If that directory already contains a completed run (`RUN_COMPLETE.json`), training is skipped and the existing features are reused. Set `RUN_TRAINING = False` to require the completed run and never train.


In [ ]:
import random
import shutil
import torch
from unittest.mock import patch
import omnixas.data.feff_graph as graph_data
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from torch.utils.data import DataLoader
from omnixas.model.m3gnet_xas import M3GNetXAS
from omnixas.data.feff_graph import (FEFFDataset, CollateGraphs, BalancedTaskBatchSampler,
    patch_matgl_gpu_constants, validate_raw_structures)

FEFF_TASKS = [f"{e}_FEFF" for e in ELEMENTS + ["Zn"]]

class LitScratch(pl.LightningModule):
    def __init__(self, model: M3GNetXAS, train_base: torch.Tensor, val_base: torch.Tensor, lr: float):
        super().__init__()
        self.model, self.lr = model, lr
        self.register_buffer("train_base", train_base)
        self.register_buffer("val_base", val_base)
        self.val_mse, self.val_task = [], []

    def step(self, batch, stage):
        graph = batch["graph"].to(self.device)
        line_graph = batch["line_graph"].to(self.device)
        site = batch["site"].to(self.device)
        pred = self.model(graph, line_graph, site)
        y = batch["y"].to(self.device)
        task = batch["task"].to(self.device)
        mse = ((pred - y) ** 2).mean(1)
        base = self.train_base if stage == "train" else self.val_base
        loss = (mse / base[task].clamp_min(1e-12)).mean() + 0.02 * (
            torch.diff(pred, dim=1) - torch.diff(y, dim=1)
        ).square().mean()
        self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True, prog_bar=True, batch_size=y.shape[0])
        if stage == "val":
            self.val_mse.append(mse.detach())
            self.val_task.append(task.detach())
        return loss

    def training_step(self, batch, _):
        return self.step(batch, "train")

    def on_validation_epoch_start(self):
        self.val_mse, self.val_task = [], []

    def validation_step(self, batch, _):
        return self.step(batch, "val")

    def on_validation_epoch_end(self):
        if self.trainer.sanity_checking or not self.val_mse:
            return
        mses, tasks = torch.cat(self.val_mse), torch.cat(self.val_task)
        rel = [
            mses[tasks == i].median() / self.val_base[i].clamp_min(1e-12)
            for i in range(len(FEFF_TASKS))
            if (tasks == i).any()
        ]
        if len(rel) != len(FEFF_TASKS):
            raise RuntimeError("Validation lacks one or more FEFF elements")
        self.log("val_balanced_rel_mse", torch.stack(rel).mean(), on_epoch=True, prog_bar=True)

    def configure_optimizers(self):
        opt = torch.optim.AdamW(
            self.parameters(),
            lr=self.lr,
            weight_decay=1e-5,
            fused=self.device.type == "cuda",
        )
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=16, min_lr=1e-6)
        return {
            "optimizer": opt,
            "lr_scheduler": {"scheduler": sched, "monitor": "val_balanced_rel_mse"},
        }



def run_universal_encoder():
    tasks = FEFF_TASKS
    workers = 0
    with patch.object(graph_data, "FEFF_TASKS", tasks):
        patch_matgl_gpu_constants()
        validate_raw_structures(REPO_ROOT, RAW_ROOT, tasks, DATASET_ROOT)
        ENCODER_RUN.mkdir(parents=True, exist_ok=False)
        random.seed(SEED)
        np.random.seed(SEED)
        torch.manual_seed(SEED)
        pl.seed_everything(SEED, workers=True)
        torch.set_float32_matmul_precision("high")
        (ENCODER_RUN / "provenance.json").write_text(json.dumps({
            "tasks": tasks, "dataset_sha256": encoder_identity, "seed": SEED,
            "model": "M3GNetXAS trained from scratch", "feature_dim": 64,
            "feature_scale": 1000, "target_scale": 1, "encoder_lr": 0.001,
            "epochs_max": 1000, "early_stop_patience": 60,
            "rows_per_element": 64, "eval_batch_size": 1024,
            "precision": "bf16-mixed", "head_dropout": 0.05,
            "selection": "validation macro relative median MSE",
        }, indent=2), encoding="utf-8")
        bases = []
        for split in ("train", "val"):
            bases.append(torch.tensor([np.median(((data[e, split][1] - data[e, "train"][1].mean(0)) ** 2).mean(1))
                                       for e in ELEMENTS + ["Zn"]]))
        train_base, val_base = bases
        model = M3GNetXAS(head_dropout=0.05)
        collate = CollateGraphs(model.encoder)
        train_ds = FEFFDataset(REPO_ROOT, RAW_ROOT, tasks, "train", DATASET_ROOT)
        counts = [sum(row[0] == task for row in train_ds.rows) for task in tasks]
        sampler = BalancedTaskBatchSampler(train_ds.rows, min(64, min(counts)), SEED)
        loader_options = {"num_workers": workers}
        if workers:
            loader_options.update(persistent_workers=True, prefetch_factor=2)
        train_loader = DataLoader(train_ds, batch_sampler=sampler, collate_fn=collate, **loader_options)
        val_ds = FEFFDataset(REPO_ROOT, RAW_ROOT, tasks, "val", DATASET_ROOT)
        val_loader = DataLoader(val_ds, batch_size=1024, collate_fn=collate, **loader_options)
        checkpoint = ModelCheckpoint(
            ENCODER_RUN / "encoder_checkpoints", filename="best-{epoch:03d}-{val_balanced_rel_mse:.5f}",
            monitor="val_balanced_rel_mse", mode="min", save_top_k=1, save_last=True)
        trainer = pl.Trainer(
            max_epochs=1000, accelerator="auto", devices=1, precision="bf16-mixed",
            callbacks=[checkpoint, EarlyStopping(monitor="val_balanced_rel_mse", patience=60, mode="min")],
            logger=CSVLogger(str(ENCODER_RUN), name="encoder_logs"), log_every_n_steps=10)
        trainer.fit(LitScratch(model, train_base, val_base, 0.001), train_loader, val_loader)
        if not checkpoint.best_model_path:
            raise RuntimeError("Encoder training produced no validation checkpoint")
        encoder_path = ENCODER_RUN / "best_encoder.ckpt"
        shutil.copy2(checkpoint.best_model_path, encoder_path)
        state = torch.load(encoder_path, map_location="cpu", weights_only=False)["state_dict"]
        model.load_state_dict({k.removeprefix("model."): v for k, v in state.items() if k.startswith("model.")})
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model.to(device).eval()
        features = ENCODER_RUN / "features"
        features.mkdir()
        with torch.inference_mode():
            for task in tqdm(tasks, desc="Export features"):
                for split in SPLITS:
                    dataset = FEFFDataset(REPO_ROOT, RAW_ROOT, [task], split, DATASET_ROOT)
                    loader = DataLoader(dataset, batch_size=1024, collate_fn=collate, **loader_options)
                    xs, ys = [], []
                    for batch in loader:
                        xs.append(model.encode(batch["graph"].to(device), batch["line_graph"].to(device),
                                               batch["site"].to(device), scaled=True).cpu().numpy())
                        ys.append(batch["y"].numpy())
                    np.savetxt(features / f"{task}_{split}_X.txt", np.concatenate(xs))
                    np.savetxt(features / f"{task}_{split}_y.txt", np.concatenate(ys))
                    ids = [f"{mid}_{site:03d}" for _, mid, site, _ in dataset.rows]
                    (features / f"{task}_{split}_ids.txt").write_text("".join(i + "\n" for i in ids), encoding="utf-8")
        for task in tasks:
            element = task.split("_", 1)[0]
            for split in SPLITS:
                X = np.loadtxt(features / f"{task}_{split}_X.txt", dtype=np.float32, ndmin=2)
                y = np.loadtxt(features / f"{task}_{split}_y.txt", dtype=np.float32, ndmin=2)
                ids = (features / f"{task}_{split}_ids.txt").read_text().splitlines()
                expected_ids, expected_y = data[element, split]
                if X.shape != (len(ids), 64) or not np.isfinite(X).all() or ids != expected_ids or not np.array_equal(y, expected_y):
                    raise ValueError(f"Exported feature mismatch: {task} {split}")
        (ENCODER_RUN / "RUN_COMPLETE.json").write_text(json.dumps({
            "status": "complete", "stage": "encoder_only", "encoder": str(encoder_path)}, indent=2), encoding="utf-8")


encoder_identity = hashlib.sha256(marker.read_bytes()).hexdigest()
encoder_marker = ENCODER_RUN / "notebook_dataset_sha256.txt"
if ENCODER_RUN.exists():
    if not (ENCODER_RUN / "RUN_COMPLETE.json").is_file():
        raise RuntimeError(f"Encoder run exists but is incomplete: {ENCODER_RUN}. Remove it to retrain.")
    if not encoder_marker.is_file() or encoder_marker.read_text().strip() != encoder_identity:
        raise ValueError("Encoder dataset provenance missing or changed; choose a new run directory")
    if json.loads((ENCODER_RUN / "RUN_COMPLETE.json").read_text()).get("stage") != "encoder_only":
        raise ValueError("Existing run is not an encoder-only run")
    print(f"Reusing completed encoder run: {ENCODER_RUN}")
elif RUN_TRAINING:
    run_universal_encoder()
    encoder_marker.write_text(encoder_identity, encoding="utf-8")
else:
    raise FileNotFoundError(f"No encoder run and RUN_TRAINING is False: {ENCODER_RUN}")

FEATURES_DIR = ENCODER_RUN / "features"
for task in [f"{e}_FEFF" for e in ELEMENTS + ["Zn"]]:
    for split in SPLITS:
        for suffix in ("X", "y", "ids"):
            path = FEATURES_DIR / f"{task}_{split}_{suffix}.txt"
            if not path.is_file():
                raise FileNotFoundError(f"Missing exported feature file: {path}")
print("encoder features:", FEATURES_DIR)


In [ ]:
# Verify the exported Zn features against the canonical dataset before any training.
for split in SPLITS:
    X = np.atleast_2d(np.loadtxt(FEATURES_DIR / f"Zn_FEFF_{split}_X.txt", dtype=np.float32))
    y = np.atleast_2d(np.loadtxt(FEATURES_DIR / f"Zn_FEFF_{split}_y.txt", dtype=np.float32))
    ids = (FEATURES_DIR / f"Zn_FEFF_{split}_ids.txt").read_text().splitlines()
    expected_ids, expected_y = data["Zn", split]
    if X.shape != (len(expected_y), 64) or y.shape != (len(expected_y), 141):
        raise ValueError(f"Bad Zn feature shapes for {split}: X{X.shape} y{y.shape}")
    if ids != expected_ids:
        raise ValueError(f"Exported Zn ids differ from dataset for split {split}")
    if not np.array_equal(y, expected_y):
        raise ValueError(f"Exported Zn targets differ from dataset for split {split}")
    if not np.isfinite(X).all():
        raise ValueError(f"Non-finite Zn features for split {split}")
print("Zn features are consistent with the canonical dataset.")


## ExpertXAS sweep on the frozen universal-encoder features

Loads the exported Zn 64D features, then trains one XASBlock head per architecture in `EXPERT_SWEEP`.

Fixed for every architecture (identical to the Zn notebook's expert branch):

- lr 0.002, Adam, plateau schedule (factor 0.5, patience 16, min_lr 1e-6)
- dropout 0.05, batch size 4096, up to 800 epochs, early stop after 60 epochs without validation improvement
- validation selection: macro-relative median MSE (single task, so identical ranking to median MSE)
- seed 43, loss-balance power 1.5 (a no-op for a single task, kept for parity)
- features, targets, and split assignment are exactly what the encoder exported; Zn only

Only `hidden_dims` changes. Architectures that already have `best.pt` and `metrics.json` under `expert_sweep/<slug>/` are skipped unless `FORCE_RESWEEP = True`. Each architecture writes its directory (checkpoints), then `expert_sweep/sweep_results.csv` is rebuilt from the per-architecture `metrics.json` files. Heads are XASBlock modules (Linear -> BatchNorm -> SiLU -> Dropout, Softplus output), same as every other head in this project.


In [ ]:
import random
import torch
from unittest.mock import patch
from omnixas.model.xasblock import XASBlock

SWEEP_X = np.atleast_2d(np.loadtxt(FEATURES_DIR / "Zn_FEFF_train_X.txt", dtype=np.float32))
SWEEP_Y = np.atleast_2d(np.loadtxt(FEATURES_DIR / "Zn_FEFF_train_y.txt", dtype=np.float32))
VAL_X = np.atleast_2d(np.loadtxt(FEATURES_DIR / "Zn_FEFF_val_X.txt", dtype=np.float32))
VAL_Y = np.atleast_2d(np.loadtxt(FEATURES_DIR / "Zn_FEFF_val_y.txt", dtype=np.float32))
if SWEEP_X.shape[1] != 64 or SWEEP_Y.shape[1] != 141:
    raise ValueError(f"Bad Zn feature shapes: X{SWEEP_X.shape} y{SWEEP_Y.shape}")
print(f"Zn sweep data: train {SWEEP_X.shape}, val {VAL_X.shape}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
Zn_train_base = float(np.median(((SWEEP_Y - SWEEP_Y.mean(0)) ** 2).mean(1)))
Zn_val_base = float(np.median(((VAL_Y - SWEEP_Y.mean(0)) ** 2).mean(1)))
if len(SWEEP_X) < 2:
    raise ValueError("Batch normalization requires at least two training rows")
print(f"Zn train baseline (median MSE): {Zn_train_base:.6f}")


def train_expert(out_dir, X, y, val_X, val_y, dims, label):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    random.seed(HEAD_SEED)
    np.random.seed(HEAD_SEED)
    torch.manual_seed(HEAD_SEED)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(HEAD_SEED)
    with patch.object(XASBlock, "DROPOUT", HEAD_DROPOUT):
        head = XASBlock(64, list(dims), 141).to(device)
    n_params = sum(p.numel() for p in head.parameters())

    Xt = torch.as_tensor(X, device=device)
    yt = torch.as_tensor(y, device=device)
    vXt = torch.as_tensor(val_X, device=device)
    base = torch.tensor([Zn_train_base], dtype=torch.float32, device=device).clamp_min(1e-12)

    def val_metric():
        head.eval()
        with torch.inference_mode():
            errors = (head(vXt) - torch.as_tensor(val_y, device=device)).square().mean(1)
            return float(torch.quantile(errors, 0.5) / base[0])

    opt = torch.optim.Adam(head.parameters(), lr=HEAD_LR, fused=device.type == "cuda")
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=16, min_lr=1e-6)
    best, stale, best_epoch, epochs_run = float("inf"), 0, -1, 0
    for epoch in tqdm(range(HEAD_EPOCHS), desc=label, unit="epoch", leave=False):
        head.train()
        batches = list(torch.randperm(len(Xt), device=device).split(HEAD_BATCH))
        if len(batches) > 1 and len(batches[-1]) == 1:
            batches[-2:] = [torch.cat(batches[-2:])]
        for index in batches:
            opt.zero_grad(set_to_none=True)
            errors = (head(Xt[index]) - yt[index]).square().mean(1)
            errors.mean().backward()  # single task: loss-balance power is a no-op
            opt.step()
        metric = val_metric()
        epochs_run = epoch + 1
        if metric < best:
            best, stale, best_epoch = metric, 0, epoch
            torch.save({"state_dict": head.state_dict(), "hidden_dims": list(dims),
                        "dropout": HEAD_DROPOUT, "epoch": epoch,
                        "val_macro_relative_median": best, "seed": HEAD_SEED,
                        "learning_rate": HEAD_LR, "head_patience": HEAD_PATIENCE},
                       out_dir / "best.pt")
        else:
            stale += 1
        sched.step(metric)
        if stale >= HEAD_PATIENCE:
            break
    # Report the plain median MSE of the selected checkpoint for interpretability.
    state = torch.load(out_dir / "best.pt", map_location=device, weights_only=False)
    head.load_state_dict(state["state_dict"])
    head.eval()
    with torch.inference_mode():
        val_errors = (head(vXt) - torch.as_tensor(val_y, device=device)).square().mean(1).cpu().numpy()
    return {
        "params": int(n_params),
        "best_val_macro_relative_median": float(best),
        "val_median_mse": float(np.median(val_errors)),
        "val_mse": float(np.mean(val_errors)),
        "val_eta": float(Zn_val_base / np.median(val_errors)) if np.median(val_errors) > 0 else float("inf"),
        "best_epoch": int(best_epoch),
        "epochs_run": int(epochs_run),
    }



In [ ]:
sweep_identity = {"encoder_sha256": sha256(ENCODER_RUN / "best_encoder.ckpt"),
                  "dataset_sha256": encoder_identity, "feature_sha256": {s: sha256(FEATURES_DIR / f"Zn_FEFF_{s}_X.txt") for s in ("train", "val")},
                  "lr": HEAD_LR, "dropout": HEAD_DROPOUT, "epochs": HEAD_EPOCHS,
                  "patience": HEAD_PATIENCE, "batch": HEAD_BATCH, "seed": HEAD_SEED,
                  "scheduler": "plateau factor=0.5 patience=16 min_lr=1e-6"}
sweep_records = []
for dims in EXPERT_SWEEP:
    slug = arch_slug(dims)
    arch_dir = SWEEP_DIR / slug
    metrics_path = arch_dir / "metrics.json"
    ckpt = arch_dir / "best.pt"
    if not FORCE_RESWEEP and ckpt.is_file() and metrics_path.is_file():
        record = json.loads(metrics_path.read_text())
        if record.get("provenance") != sweep_identity or record.get("checkpoint_sha256") != sha256(ckpt):
            raise ValueError(f"Stale or incompatible sweep artifacts: {arch_dir}; use a new output directory")
        record["slug"] = slug
        record["hidden_dims"] = "-".join(str(d) for d in dims)
        sweep_records.append(record)
        print(f"skip {slug} (exists): val_median_mse={record['val_median_mse']:.4e}")
        continue
    if not RUN_TRAINING:
        print(f"skip {slug}: RUN_TRAINING is False and no completed run exists")
        continue
    t0 = time.time()
    result = train_expert(arch_dir, SWEEP_X, SWEEP_Y, VAL_X, VAL_Y, dims, f"Expert {slug}")
    result.update({"provenance": sweep_identity, "checkpoint_sha256": sha256(ckpt), "slug": slug, "hidden_dims": "-".join(str(d) for d in dims),
                   "seed": HEAD_SEED, "lr": HEAD_LR, "dropout": HEAD_DROPOUT,
                   "epochs_max": HEAD_EPOCHS, "patience": HEAD_PATIENCE,
                   "wall_time_s": round(time.time() - t0, 1)})
    metrics_path.write_text(json.dumps(result, indent=2))
    sweep_records.append(result)
    print(f"done {slug}: val_median_mse={result['val_median_mse']:.4e} "
          f"({result['epochs_run']} epochs, {result['wall_time_s']}s)", flush=True)

df = pd.DataFrame(sweep_records)
if len(df):
    df = df.sort_values("val_median_mse").reset_index(drop=True)
    SWEEP_DIR.mkdir(parents=True, exist_ok=True)
    df.to_csv(SWEEP_DIR / "sweep_results.csv", index=False)
    print(f"wrote {SWEEP_DIR / 'sweep_results.csv'}")
    display(df)
else:
    print("No sweep records. Set RUN_TRAINING = True to run the sweep.")


## Sweep results and winner

Sorted by validation median MSE. The winner is the first row. The test split is still untouched at this point.


In [ ]:
results_path = SWEEP_DIR / "sweep_results.csv"
if not results_path.is_file():
    raise FileNotFoundError(f"No sweep results yet: {results_path}. Run the sweep cell first.")
df = pd.read_csv(results_path).sort_values("val_median_mse").reset_index(drop=True)
expected_slugs = {arch_slug(dims) for dims in EXPERT_SWEEP}
if set(df["slug"]) != expected_slugs or len(df) != len(expected_slugs):
    raise ValueError("Sweep is incomplete or differs from EXPERT_SWEEP; finish the sweep before selecting a winner")
best = df.iloc[0]
print("winner:", best["hidden_dims"], "val_median_mse=", f"{best['val_median_mse']:.6e}")
display(df[["hidden_dims", "params", "val_median_mse", "val_mse", "val_eta", "best_epoch", "epochs_run"]].round(6))

fig, ax = plt.subplots(figsize=(12, 5))
order = df.sort_values("val_median_mse")
ax.bar(range(len(order)), order["val_median_mse"], color="steelblue")
ax.set_xticks(range(len(order)))
ax.set_xticklabels(order["hidden_dims"], rotation=90, fontsize=7)
ax.set_yscale("log")
ax.set_ylabel("Validation median MSE (log scale)")
ax.set_xlabel("hidden_dims")
ax.set_title("ExpertXAS architecture sweep: validation median MSE (lower is better)")
ax.axhline(Zn_val_base, color="gray", ls="--", lw=1, label="train-mean baseline on validation")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "sweep_val_median_mse.png", dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)

winner = {"hidden_dims": [int(v) for v in str(best["hidden_dims"]).split("-")],
          "slug": str(best["slug"]),
          "selection": "validation_median_mse",
          "val_median_mse": float(best["val_median_mse"])}
(OUTPUT_ROOT / "winner.json").write_text(json.dumps(winner, indent=2))
print("winner written to", OUTPUT_ROOT / "winner.json")


## Test evaluation and noise-ratio chart

The architecture and checkpoint are selected using validation median MSE. The winner is then evaluated on the Zn test set. The baseline is the mean Zn training spectrum evaluated on the test set.

Each point represents one test spectrum. The x coordinate is the mean squared error between its prediction and target across 141 energies. For the y coordinate, smooth the target with a Gaussian kernel of standard deviation 0.5 eV, using edge padding. Subtract the smooth curve from the target, then divide the residual standard deviation by the target standard deviation. A constant spectrum is assigned zero. The ratio is dimensionless and describes high-frequency variation, not a measured noise level: real spectral features and interpolation can also contribute. Spearman correlation uses average ranks for ties. This diagnostic does not establish causation.


In [ ]:
winner = json.loads((OUTPUT_ROOT / "winner.json").read_text())
dims = winner["hidden_dims"]
best_ckpt = SWEEP_DIR / winner["slug"] / "best.pt"
if not best_ckpt.is_file():
    raise FileNotFoundError(f"Missing winner checkpoint: {best_ckpt}")

import torch
from omnixas.model.xasblock import XASBlock

test_X = np.atleast_2d(np.loadtxt(FEATURES_DIR / "Zn_FEFF_test_X.txt", dtype=np.float32))
test_y = np.atleast_2d(np.loadtxt(FEATURES_DIR / "Zn_FEFF_test_y.txt", dtype=np.float32))
expected_ids, expected_y = data["Zn", "test"]
saved_ids = (FEATURES_DIR / "Zn_FEFF_test_ids.txt").read_text().splitlines()
if saved_ids != expected_ids or not np.array_equal(test_y, expected_y):
    raise ValueError("Test features do not match the canonical dataset")

state = torch.load(best_ckpt, map_location="cpu", weights_only=False)
with patch.object(XASBlock, "DROPOUT", HEAD_DROPOUT):
    head = XASBlock(64, dims, 141)
head.load_state_dict(state["state_dict"])
head.to(device).eval()
with torch.inference_mode():
    pred = torch.cat([head(x.to(device)).cpu() for x in torch.from_numpy(test_X).split(HEAD_BATCH)]).numpy()

metrics = score_predictions(pred, test_y, SWEEP_Y)
np.savez_compressed(OUTPUT_ROOT / "winner_test_predictions.npz", predictions=pred, targets=test_y)
(OUTPUT_ROOT / "winner_test_metrics.json").write_text(json.dumps({
    "hidden_dims": dims, "slug": winner["slug"], **metrics}, indent=2))
print(f"winner {dims} test: mse={metrics['mse']:.6e} median_mse={metrics['median_mse']:.6e} eta={metrics['eta']:.3f}")

per_spectrum_mse = ((pred - test_y) ** 2).mean(axis=1)
ratios = np.array([noise_ratio(y) for y in test_y])
chart = pd.DataFrame({"id": expected_ids, "mse": per_spectrum_mse, "noise_ratio": ratios})
chart[["material_id", "site"]] = chart["id"].str.rsplit("_", n=1, expand=True)
chart["element"] = "Zn"
chart.to_csv(OUTPUT_ROOT / "winner_test_noise_ratio.csv", index=False)
rho = spearman(per_spectrum_mse, ratios)
print(f"spectra: {len(chart)}  Spearman rho(mse, noise_ratio) = {rho:.3f}")

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(per_spectrum_mse, ratios, s=18, alpha=0.6, color="tab:blue")
ax.set_xlabel("Per-spectrum MSE (test)")
ax.set_ylabel("Noise ratio  std(y - smooth(y)) / std(y),  sigma = 0.5 eV")
ax.set_title(f"Zn test: winner {'-'.join(str(d) for d in dims)}  (Spearman rho = {rho:.2f}, n = {len(chart)})")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "winner_test_mse_vs_noise_ratio.png", dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)


## Examples by test MSE percentile

The first grid shows target and predicted spectra nearest the 5th through 95th percentiles of test MSE. Labels include Zn, the Materials Project ID, absorbing-site index, percentile, and MSE. The second grid shows nine different materials nearest the 95th percentile of test MSE. Each panel projects the source unit cell onto the Cartesian x-y plane. Marker color identifies each element, and the ring marks the absorbing Zn site. Overlapping atoms can obscure one another in this projection. These plots describe the already selected winner and must not be used to choose a model.


In [ ]:
from matplotlib.lines import Line2D
from omnixas.data.feff_graph import structure_path, parse_feff_structure
from pymatgen.core import Structure

percentiles = np.linspace(5, 95, 9)
selected = []
for q in percentiles:
    target_error = np.percentile(per_spectrum_mse, q)
    order = np.argsort(np.abs(per_spectrum_mse - target_error), kind="stable")
    index = next((int(i) for i in order if int(i) not in selected), None)
    if index is None:
        raise ValueError("At least nine test spectra are needed for the percentile grid")
    selected.append(index)

fig, axes = plt.subplots(3, 3, figsize=(14, 10), sharex=True)
for ax, q, i in zip(axes.flat, percentiles, selected):
    ax.plot(ENERGY_GRID, test_y[i], label="Target", lw=1.5)
    ax.plot(ENERGY_GRID, pred[i], label="Prediction", lw=1.2)
    ax.set_title(f"Zn {chart.material_id.iloc[i]} site {chart.site.iloc[i]} | {q:.0f}th percentile\nMSE {per_spectrum_mse[i]:.3g}", fontsize=9)
    ax.set_xlabel("Energy (eV)")
    ax.set_ylabel("Intensity (unscaled)")
    ax.grid(alpha=0.2)
axes.flat[0].legend(fontsize=8)
fig.suptitle("Zn test spectra by MSE percentile")
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "winner_test_spectra_percentiles.png", dpi=180)
display(fig)
plt.close(fig)

q95 = float(np.percentile(per_spectrum_mse, 95))
order = np.argsort(np.abs(per_spectrum_mse - q95), kind="stable")
near95, seen = [], set()
for i in order:
    mid = chart.material_id.iloc[i]
    if mid not in seen:
        seen.add(mid)
        near95.append(int(i))
    if len(near95) == 9:
        break
if len(near95) < 9:
    raise ValueError("At least nine distinct test materials are needed for the structure grid")

structures = []
for i in near95:
    mid, site = chart.material_id.iloc[i], int(chart.site.iloc[i])
    path = structure_path(RAW_ROOT, "Zn_FEFF", mid, site)
    if not path.is_file():
        raise FileNotFoundError(f"Missing source structure: {path}")
    structure = Structure.from_file(path) if path.name == "POSCAR" else parse_feff_structure(path)
    if site >= len(structure) or structure[site].specie.symbol != "Zn":
        raise ValueError(f"Absorbing site does not identify Zn: {mid}, site {site}")
    structures.append(structure)
symbols = sorted({site.specie.symbol for structure in structures for site in structure})
colors = {symbol: plt.get_cmap("tab20")(i % 20) for i, symbol in enumerate(symbols)}
fig, axes = plt.subplots(3, 3, figsize=(14, 12))
for ax, i, structure in zip(axes.flat, near95, structures):
    xyz = structure.cart_coords
    for symbol in sorted({site.specie.symbol for site in structure}):
        mask = np.array([site.specie.symbol == symbol for site in structure])
        ax.scatter(xyz[mask, 0], xyz[mask, 1], s=70, c=[colors[symbol]], edgecolors="black", linewidths=0.4)
    site = int(chart.site.iloc[i])
    ax.scatter(xyz[site, 0], xyz[site, 1], s=170, facecolors="none", edgecolors="red", linewidths=2)
    ax.set_title(f"Zn {chart.material_id.iloc[i]} site {site} | MSE {per_spectrum_mse[i]:.3g}", fontsize=9)
    ax.set_aspect("equal", adjustable="datalim")
    ax.set_xlabel("Cartesian x (Å)")
    ax.set_ylabel("Cartesian y (Å)")
legend = [Line2D([0], [0], marker="o", color="none", markerfacecolor=colors[s], markeredgecolor="black", label=s) for s in symbols]
legend.append(Line2D([0], [0], marker="o", color="red", markerfacecolor="none", label="Absorbing Zn"))
fig.legend(handles=legend, loc="lower center", ncol=min(8, len(legend)))
fig.suptitle(f"Distinct Zn test materials nearest 95th-percentile MSE ({q95:.3g})")
fig.tight_layout(rect=(0, 0.06, 1, 0.96))
fig.savefig(OUTPUT_ROOT / "winner_test_structures_near_p95.png", dpi=180)
display(fig)
plt.close(fig)


## Artifacts

Written under `output/training/zn_expert_sweep/` (nothing is committed to git):

- `zn9_universal_encoder/` — universal encoder run (checkpoint, features, pipeline provenance)
- `expert_sweep/<dims>/best.pt` and `metrics.json` — one directory per architecture
- `expert_sweep/sweep_results.csv` — all architectures, sorted by validation median MSE
- `winner_test_spectra_percentiles.png` and `winner_test_structures_near_p95.png` – test examples
- `sweep_val_median_mse.png` — sweep comparison chart
- `winner.json`, `winner_test_metrics.json`, `winner_test_predictions.npz`, `winner_test_noise_ratio.csv`, `winner_test_mse_vs_noise_ratio.png`

The noise ratio measures high-frequency variation, which can include physical features. Interpret the chart as a descriptive diagnostic.

Exploratory result on unscaled Zn Figshare data. Not comparable to published OmniXAS numbers.
